# <u>Principal Component Analysis</u>

## Topics

* [1. What is Dimensionality Reduction?](#what)
* [2. What is PCA?](#pca)
* [3. Maximize Variance](#var)
* [4. Key Results](#res)
* [5. Why Eigenvectors?](#eigenvectors)
* [6. PCA Algorithm](#algorithm)
* [7. Interpretation](#interpretation)
* [8. Additional Notes](#notes)
* [9. PCA library](#library)

In [4]:
import numpy as np # for math and random numbers
import plotly.express as px # for plotting
import plotly.graph_objects as go # for plotting
from sklearn.decomposition import PCA # for principal component analysis
from sklearn.datasets import make_regression # make data for regression
print("Setup complete")

Setup complete


<a class="anchor" id="what"></a>
# 1. What is Dimensionality Reduction?

- **Goal:** Represent high-dimensional data ($D$ dimensions) in a lower-dimensional ($d$ dimensions) space while preserving as much variance (information) as possible

- Benefits:
    - Simplifies data
    - Can improve classification/regression

- Approaches:
    - Drop dimensions (naive)
    - Drop low-variance dimensions
    - **PCA (better): Rotate data first, then reduce dimensions**

- Key idea:
    - Original variables may be correlated
    - PCA creates new variables called principal components (PCs) that are:
        - Linear combinations of the original variables
        - Uncorrelated (orthogonal)
        - Ordered by importance (variance explained)

&#128073; Interpretation:

- PC1 = direction of maximum variance
- PC2 = next largest variance, orthogonal to PC1
- Dimensionality reduction:
    - Keep only the first k PCs (Principal Components)
    - Project data onto them
    - Preserve as much variance as possible

<a class="anchor" id="pca"></a>
# 2. What is PCA?

- PCA is essentially a **rotation of the coordinate system**
- Then:
    - Find the axis/direction where data spreads the most
    - Project onto that axis
- New axes (principal components):
    - Capture the **most important patterns (variance)** in data
- Result: Data is expressed as **linear combinations of original features** in lower dimensions



Two cases:

<div style="display:flex; gap:20px;">
<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--color:white;-->
width:50%;
">


<h5 style="text-align:center;"><b>(a) Uncorrelated variables</b></h5>

- Data is most spread along one of the already existing axis
- Just keep the variable with highest variance

<p align="center">
<img src="pics/uncorrelated.png" width="600"/>
</p>

If we were to project onto axis with lowest variance (here $x_2$) data points would overlap too much resulting in loss of information about true spread of data.

<p align="center">
<img src="pics/1.png" width="600"/>
</p>

However in above example projecting onto $x_2$ would result in easier classification of red and green points.

</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--color:white;-->
width:50%;
">

<h5 style="text-align:center;"><b>(b) Correlated variables</b></h5>


- Rotate axes $\rightarrow$ find better directions (PCs)
- Then drop low-variance directions

<p align="center">
<img src="pics/correlated.png" width="600"/>
</p>

</div>
</div>

### General Procedure
- Rotate original $D$-dimensional coordinate system until first Principal component which explains most of the variation is found
- Fix that first PC and continue rotating remaining $D-1$ corrdinates until second PC (which is orthogonal to the first PC) is found that explains most of the remaining variation, etc
- Reduce dimensions by projecting points onto the first $d < D$ principal components


<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/2.png" width="550"/>
  <img src="pics/3.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/4.png" width="550"/>
  <img src="pics/5.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/6.png" width="550"/>
  <img src="pics/7.png" width="550"/>
</div>

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/8.png" width="550"/>
  <img src="pics/best.png" width="550"/>
</div>

We see that rotating finally yielded a PC whit the largest amount of variance (1.63) when projected onto

---

<div style="display: flex; justify-content: center; gap: 5px;">
  <img src="pics/best1.png" width="550"/>
  <img src="pics/best2.png" width="550"/>
</div>

---
<p align="center">
<img src="pics/9.png" width="600"/>
</p>


<a class="anchor" id="var"></a>
# 3. Maximize Variance

Given $n$ data points $x_i \in \mathbb{R}^D$ $(i=1,\ldots,n)$ collected inside a matrix $X=[x_1 \hspace{0.2cm} x_2 \hspace{0.2cm} ... \hspace{0.2cm} x_n] \in \mathbb{R}^{D \times n}$ and center $X$ to have zero mean.

Goal: Represent high dimensional data $X$ in lower dimensional space that keeps most of the properties (here variance) denoted as $Z=[z_1 \hspace{0.2cm} z_2 \hspace{0.2cm} ... \hspace{0.2cm} z_n]\in \mathbb{R}^{d \times n}$ with a new coordinate system (with fewer axes) by finding a map

$$
f: \mathbb{R}^D \rightarrow \mathbb{R}^d
$$

- Define a new coordinate system by rotation of data
- Represent a rotation by a unitary (orthonormal) matrix $W \in \mathbb{R}^{D \times d}$ meaning $W^\top W = WW^\top = I$
- However since ususally $d \ll D$ $W$ is rectangular and only $W^\top W=I$ holds and possibly $WW^\top \neq I$
- Columns of $W$ are the new coordinate axis
- linear map is $Z=W^\top X$ and $W$ as parameter
- Find direction $w \in \mathbb{R}^D$ of maximum variance with respect to $w$
$$
\max_w \frac{1}{n} \lVert w^\top X \rVert^2=\frac{1}{n} \sum_{i=1}^n (w^\top x_i)^2 \quad \text{ such that } \lVert w \rVert^2 = 1 
$$

- normalization constraint $\lVert w \rVert^2 = 1$ is required for identifiability reasons, otherwise we can maximize variance by increasing $w$ 
- projected data also has zero mean
- Find $d$ directions that maximize variance
$$
\max_W \lVert W^\top X \rVert^2= \sum_{i=1}^n (W^\top x_i)^2 \quad \text{ such that } W^\top W = I 
$$
- also $WW^\top \neq I$
- Lagrangian

$$
\begin{align*}
L(w_1) &= \frac{1}{n} \lVert w_1^\top X \rVert^2 + \lambda (\lVert w_1 \rVert^2 -1) \hspace{1 mm} \mid \hspace{1 mm} w_1^\top X \text{ is a row vector so }\lVert w_1^\top X \rVert^2=w_1^\top X(w_1^\top X)^\top  \\
&= \frac{1}{n} w_1^\top XX^\top w_1 + \lambda (w_1^\top w_1 -1) \\ 
\frac{1}{n} XX^\top =: \Sigma &\text{ is gaussian MLE estimator of covariance matrix of data $X$ with zero mean} \\
&=  \underbrace{w_1^\top \Sigma w_1}_{\text{objective}} + \lambda (w_1^\top w_1 -1) \\ 
\end{align*}
$$
- Setting derivatives to zero
$$
\begin{align*}
dL(w_1) =  2(w_1^\top \Sigma - \lambda w_1)dw_1 &= 0 \\ \Leftrightarrow
2w_1^\top \Sigma &= 2\lambda w_1 \\ \Leftrightarrow
\Sigma w_1 &= \lambda w_1 \\ 
\text{Thus $w_1$ is an eigenvector} &\text{ of $\Sigma$ and $\lambda$ the corresponding eigenvalue} \\
\underbrace{w_1^\top \Sigma w_1}_{\text{objective}} &= \lambda w_1^\top w_1 = \lambda \hspace{1 mm} \mid \hspace{1 mm} \text{since } w_1^\top w_1 = 1 \text{ is our constraint} \\
\text{So we maximize $\lambda$} &\text{ and choose $w_1$ to be the eigenvector of the largest eigenvalue denoted as $\lambda_1$}
\end{align*}
$$

- Be orthogonal to the first direction and normalized
$$
w_1^\top w_2 = 0 \hspace{2cm} w_2^\top w_2 = 1
$$

- Lagrangian
$$
\begin{align*}
L(w_2) &= w_2^\top \Sigma w_2 - \lambda(w_2^\top w_2 - 1) - \mathbb{v}(w_1^\top w_2 - 0) \\ 
&=w_2^\top \Sigma w_2 - \lambda w_2^\top w_2 - \lambda - \mathbb{v}w_1^\top w_2
\end{align*}
$$

- Setting derivatives to zero
$$
\begin{align*}
L(w_2) = 2\Sigma w_2 - 2 \lambda w_2 -\mathcal{v}w_1 &= 0 \\
\text{Multiply from}& \text{ left with } w_1^\top \\ \Leftrightarrow
2 w_1^\top \Sigma w_2 - 2 \lambda w_1^\top w_2 -\mathcal{v} w_1^\top w_1 &= 0  \hspace{1 mm} \mid \hspace{1 mm} w_1^\top w_2 = 0,w_1^\top w_1 = 1 \\ \Leftrightarrow
2 w_1^\top \Sigma w_2 -\mathcal{v} &= 0 \hspace{1 mm} \mid \hspace{1 mm} w_1^\top \Sigma = \lambda_1 w_1^\top \\ \Leftrightarrow
2 \lambda_1 w_1^\top w_2 - \mathcal{v} &= 0 \hspace{0.2cm} \text{| } w_1^\top w_2 = 0 \\ \Leftrightarrow
\mathcal{v} &= 0 \\ 
\Rightarrow 2\Sigma w_2 - 2 \lambda w_2 -\mathcal{v}w_1 &= 0 \\ \Leftrightarrow 
\Sigma w_2  &= \lambda w_2 \\
w_2 \text{ should be eigenvector to second } &\text{largest eigenvalue }\lambda_2
\end{align*}
$$
 

#### Eigenvalue Decomposition

Any symmetric matrix $A$ can be decomposed into $$A=V \Lambda V^\top$$

- $\Lambda$: diagonal matrix with entries along diagonal called eigenvalues
- $V$: unitary matrix so $VV^\top = V^\top V=I$ and columns of $V$ are eigenvectors
- The factorization $A=V \Lambda V^\top$ implies $Av = \lambda v$


In [2]:
np.random.seed(1239)
D, n = 2, 100

# mean and covariance (positive correlation)
mean = [3, 4]
cov = [[6, 3],
       [3, 2]] 

X = np.random.multivariate_normal(mean, cov, size=n)

fig = px.scatter(x=X[:, 0], y=X[:, 1],title="Uncentered data",labels={"x":"x1","y":"x2"})
fig.show()

# Center data
X_centered = X - np.mean(X, axis=0)

# Eigen decomposition
Lambda, V = np.linalg.eig(cov)

# sort by descending eigenvalues
idx = np.argsort(Lambda)[::-1]
Lambda = Lambda[idx]
V = V[:, idx]

# Plot 
fig = px.scatter(x=X_centered[:, 0],y=X_centered[:, 1],title="Centered data with principal directions",labels={"x": "x1", "y": "x2"})

# Add axes at 0 
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.add_vline(x=0, line_dash="dash", line_color="black")

# Add eigenvectors 
origin = np.array([0, 0])

for i in range(2):
    vec = V[:, i]
    length = np.sqrt(Lambda[i]) * 2 # scale for visibility
    fig.add_trace(go.Scatter(x=[origin[0], vec[0]*length],y=[origin[1], vec[1]*length],mode="lines+markers",name=f"PC{i+1} (λ={Lambda[i]:.2f})"))

fig.show()

<a class="anchor" id="res"></a>
# 4. Key Results

- The optimal directions (principal components) are:
    - **Eigenvectors of the covariance matrix**
- Ordered by importance:
    - Largest eigenvalue $\rightarrow$ most variance $\rightarrow$ most important direction


<a class="anchor" id="eigenvectors"></a>
# 5. Why Eigenvectors?

- Maximizing variance leads to:
    - Optimization problem $\rightarrow$ solved with Lagrange multipliers
- Result:
    - Solution must satisfy eigenvalue equation
    - Best direction = eigenvector with **largest eigenvalue**
    - **each eigenvalue (like $\lambda_2$) corresponds to the variance of the data along its associated eigenvector (PC2).**

**Choosing direction of largest variance as new axis = choosing eigenvector of largest eigenvalue**

$$\text{Goal: Project onto PC1 in 1D}$$

<p align="center">
<img src="pca\17.jpg" width="600"/>
</p>

____


$$\text{Directions of variance}$$

<p align="center">
<img src="pca\18.jpg" width="600"/>
</p>

____


$$\text{Directions of variance}$$

<p align="center">
<img src="pca\19.jpg" width="600"/>
</p>

____



$$\text{Why choose direction of largest variance ?}$$

<p align="center">
<img src="pca\20.jpg" width="600"/>
</p>


- Axis 1 &#9989;: We get a faithful representation of original data. Points *Rome* and *NYC* are far apart like in original data
- Axis 2: We get a bad representation of original data. Points are too close and we cannot distingiush well between *Rome* and *NYC*

____



<p align="center">
<img src="pca\21.jpg" width="600"/>
</p>

$$ \text{Amount of remaining information: }\frac{\lambda_1 + ... + \lambda_d}{\sum_{i=1}^D \lambda_i} \ge k \hspace{0.2cm} (\text{i.e. k = 0.9 = 90 \%})   $$

____


<p align="center">
<img src="pca\22.jpg" width="600"/>
</p>

- Project onto axis with most variance
- Then flip axis

____

$$\text{Outlook: Nonlinear PCA}$$

<p align="center">
<img src="pca\23.jpg" width="600"/>
</p>

- PCA is a linear method
- For nonlinear data the direction of largest variance is a curved axis
- **Use kernel PCA (kPCA)** 


In [3]:
np.random.seed(1239)
D, n = 2, 100

# mean and covariance (positive correlation)
mean = [3, 4]
cov = [[6, 3],
       [3, 2]] 

X = np.random.multivariate_normal(mean, cov, size=n)

#fig = px.scatter(x=X[:, 0], y=X[:, 1],title="Uncentered data",labels={"x":"x1","y":"x2"})
#fig.show()

# Center data
X_centered = X - np.mean(X, axis=0)

# Eigen decomposition
Lambda, V = np.linalg.eig(cov)

# sort by descending eigenvalues
idx = np.argsort(Lambda)[::-1]
Lambda = Lambda[idx]
V = V[:, idx]

# Plot 
fig = px.scatter(x=X_centered[:, 0],y=X_centered[:, 1],title="Centered data with principal directions",labels={"x": "x1", "y": "x2"})

# Add axes at 0 
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.add_vline(x=0, line_dash="dash", line_color="black")

# Add eigenvectors 
origin = np.array([0, 0])

for i in range(2):
    vec = V[:, i]
    length = np.sqrt(Lambda[i]) * 2 # scale for visibility
    fig.add_trace(go.Scatter(x=[origin[0], vec[0]*length],y=[origin[1], vec[1]*length],mode="lines+markers",name=f"PC{i+1} (λ={Lambda[i]:.2f})"))

fig.show()

# eigenvalues already sorted descending
total_var = np.sum(Lambda)

explained_ratio = Lambda / total_var
cumulative_ratio = np.cumsum(explained_ratio)

#print("Eigenvalues:", Lambda)
#print("Explained variance ratio:", explained_ratio)
#print("Cumulative variance:", cumulative_ratio)

# Bar plot 
fig = go.Figure()
fig.add_trace(go.Bar(x=[f"PC{i+1}" for i in range(len(Lambda))],y=explained_ratio,name="Explained variance"))
# cumulative line
#fig.add_trace(go.Scatter(x=[f"PC{i+1}" for i in range(len(Lambda))],y=cumulative_ratio,mode="lines+markers",name="Cumulative variance"))
fig.update_layout(title="Explained Variance by Principal Components",xaxis_title="Principal Components",yaxis_title="Variance Ratio")

fig.show()

<a class="anchor" id="algorithm"></a>
# 6. PCA Algorithm

**Algorithm** (based on EVD of covariance matrix)

1. Given data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{D\times n}$, assume mean is already removed or remove it
2. Calculate sample covariance matrix $\Sigma = \frac{1}{n} \sum_{i=1}^n x_i x_i^\top = \frac{1}{n} XX^\top$
3. Calculate eigenvector decomposition (EVD) $\Sigma = V \Lambda V^\top$
4. $V=[\mathcal{v_1} \hspace{0.2cm} \mathcal{v_2} \hspace{0.2cm} ... \hspace{0.2cm} \mathcal{v_D}]$ contains eigenvectors as columns and $\Lambda = \begin{bmatrix} \lambda_1 & 0 & ... & 0 \\ 0 & \lambda_2 & ... & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & ... & \lambda_D \end{bmatrix}$ contains corresponding eigenvalues $\lambda_1 \ge \ldots \ge \lambda_D$
5. Pick $d$ largest eigenvalues carrying most of the variance
6. Project data onto $V_d=[\mathcal{v_1} \hspace{0.2cm} \ldots \hspace{0.2cm} \mathcal{v_d}]$, i.e., $Z=V_d^\top X$
    - Let $\Sigma = \text{Cov}(X)=V \Lambda V^\top$
$$
\begin{align*}
\text{Cov}(Z) &= \text{Cov}(V^\top X) \\
&= V^\top \text{Cov}(X) V \\
&= V^\top \Sigma V \\
&= V^\top (V \Lambda V^\top) V \\
&= V^\top V \Lambda V^\top V \hspace{1 mm} \mid \hspace{1 mm} V^\top V = I \\
&= \Lambda \text{ diagonal covaraince matrix $\Rightarrow$ no correlations between data in projected space}
\end{align*}
$$
7. Possibly rescale axes $Z=\Lambda_d^{-0.5}V_d^\top X$ with $\Lambda_d$ containing $\lambda_1,..,\lambda_d$  (aka whitening) 

Whitening enforces:

- zero mean
- unit variance
- no correlations

Without whitening:
- Variances = eigenvalues
- PC1 dominates numerically
- Algorithms may focus almost entirely on high-variance directions

With whitening:
- All dimensions are treated equally
- Variances = 1

____

**Algorithm** (based on SVD of data matrix)

1. Given data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{D\times n}$, assume mean is already removed
2. Calculate Singular value decomposition (SVD) of data matrix $X = USV^\top$
3. $U=[u_1,...,u_D]$ and $S=diag(s_1,...,s_{\min(D,n)})$ with $s_1 \ge \ldots \ge s_{\min(D,n)}$
4. Pick $d$ largest singular values carrying most of the variance
5. Project data onto $U_d=[u_1,...,u_d]$, i.e., $Z=U_d^\top X$


<p align="center">
<img src="pca\36.jpg" width="600"/>
</p>


<p align="center">
<img src="pca\37.jpg" width="600"/>
</p>



<p align="center">
<img src="pca\38.jpg" width="600"/>
</p>



<p align="center">
<img src="pca\39.jpg" width="600"/>
</p>


<p align="center">
<img src="pca\40.jpg" width="600"/>
</p>


<p align="center">
<img src="pca\41.jpg" width="600"/>
</p>


<p align="center">
<img src="pca\42.jpg" width="600"/>
</p>


In [4]:
def pca(X, d=None, whiten=False, return_all=False):
    """
    PCA via eigenvalue decomposition of covariance matrix

    Parameters
    ----------
    X : array, shape (D, n)
        Data matrix (features x samples)
    d : int or None
        Number of components to keep (if None, keep all)
    whiten : bool
        Whether to whiten the projected data
    return_all : bool
        If True, return eigenvalues and eigenvectors as well

    Returns
    -------
    Z : array, shape (d, n)
        Projected data
    V_d : array, shape (D, d)
        Principal directions
    (optional) Lambda, V
    """

    # 1. Center data
    X_centered = X - np.mean(X, axis=1, keepdims=True)

    # 2. Covariance matrix
    n = X.shape[1]
    Sigma = (1 / n) * X_centered @ X_centered.T

    # 3. Eigen decomposition
    Lambda, V = np.linalg.eigh(Sigma)

    # 4. Sort descending
    idx = np.argsort(Lambda)[::-1]
    Lambda = Lambda[idx]
    V = V[:, idx]

    # 5. Select top d
    if d is None:
        d = X.shape[0]

    V_d = V[:, :d]
    Lambda_d = Lambda[:d]

    # 6. Project
    Z = V_d.T @ X_centered

    # 7. Whitening (optional)
    if whiten:
        Z = np.diag(1 / np.sqrt(Lambda_d)) @ Z

    if return_all:
        return Z, V_d, Lambda, V
    else:
        return Z, V_d
    

# convert your data to (D, n)
X_T = X_centered.T   # if X was (n, D)
Z, V_d, Lambda, V = pca(X_T, d=2, whiten=False, return_all=True)
print("Eigenvalues:",Lambda)
print("Eigenvectors:\n",V,"\n")
print("Eigenvectors corresponding to d biggest eigenvalue:\n",V_d,"\n")
print("Projected data covariance (eigenvalues as variances):\n",np.cov(Z),"\n")
Z, V_d, Lambda, V = pca(X_T, d=2, whiten=True, return_all=True)
print("Projected data covariance (after whitening variances become 1):\n",np.cov(Z),"\n")

Eigenvalues: [7.62632966 0.3175193 ]
Eigenvectors:
 [[-0.87816981  0.47834901]
 [-0.47834901 -0.87816981]] 

Eigenvectors corresponding to d biggest eigenvalue:
 [[-0.87816981  0.47834901]
 [-0.47834901 -0.87816981]] 

Projected data covariance (eigenvalues as variances):
 [[7.70336330e+00 1.40155577e-15]
 [1.40155577e-15 3.20726566e-01]] 

Projected data covariance (after whitening variances become 1):
 [[1.01010101e+00 8.98352002e-16]
 [8.98352002e-16 1.01010101e+00]] 



In [9]:
np.random.seed(1355)
D, n = 3, 50

mean = [3, 4, 5]

# positive correlations in 3D
cov = [[6, 3, 2],
       [3, 2, 1],
       [2, 1, 1]]

X = np.random.multivariate_normal(mean, cov, size=n)

# center data
X_centered = X - np.mean(X, axis=0)

X_T = X_centered.T # if X was (n, D)
Z, V_d, Lambda, V = pca(X_T, d=2, whiten=False, return_all=True)

fig = px.scatter_3d(x=X_centered[:,0],y=X_centered[:,1],z=X_centered[:,2],title="Centered 3D data with principal directions")


# plot eigenvectors
origin = np.array([0,0,0]) # origin
colors = ["red", "green", "blue"]
for i in range(3):
    vec = V[:, i]
    length = np.sqrt(Lambda[i]) * 3  # slightly larger scaling
    fig.add_trace(go.Scatter3d(
        x=[0, vec[0]*length],y=[0, vec[1]*length],z=[0, vec[2]*length],
        mode="lines+markers",line=dict(width=6, color=colors[i]),marker=dict(size=4, color=colors[i]),name=f"PC{i+1} (lambda={Lambda[i]:.2f})"))

fig.show()

total_var = np.sum(Lambda)
explained_ratio = Lambda / total_var
cumulative_ratio = np.cumsum(explained_ratio)

fig = go.Figure()

fig.add_trace(go.Bar(x=[f"PC{i+1}" for i in range(3)],y=explained_ratio,name="Explained variance"))
fig.add_trace(go.Scatter(x=[f"PC{i+1}" for i in range(3)],y=cumulative_ratio,mode="lines+markers",name="Cumulative variance"))
fig.add_hline(y=0.9, line_dash="dash") # 90% threshold
fig.update_layout(title="Explained Variance by PCs",yaxis_title="Variance Ratio")
fig.show()

Z_plot = Z.T  # shape (n, d)
fig = px.scatter(x=Z_plot[:,0],y=Z_plot[:,1],title="Projection onto first 2 principal components",labels={"x":"PC1","y":"PC2"})

# add axes
fig.add_hline(y=0, line_dash="dash")
fig.add_vline(x=0, line_dash="dash")

fig.show()

In [18]:
# Generate toy data
np.random.seed(1439)
D, n = 2, 75

mean = [4, 3]
cov = [[6, 2],
       [2, 2]]

X = np.random.multivariate_normal(mean, cov, size=n)

# Center data
X_centered = X - np.mean(X, axis=0)

X_for_pca = X_centered.T  # shape (2, n)
Z, V_d, Lambda, V = pca(X_for_pca, return_all=True)


# Compare eigenvalues
mean_squared_projections = np.mean(Z**2, axis=1)

print("Eigenvalues:")
print(Lambda)

print("\nMean squared projections:")
print(mean_squared_projections)


fig = px.scatter(x=X_centered[:, 0],y=X_centered[:, 1],title="Centered data with principal directions",labels={"x": "x1", "y": "x2"})

# Add eigenvectors as lines
origin = np.array([0, 0])

for i in range(2):
    vec = V[:, i]
    scale = np.sqrt(Lambda[i]) * 2  # scale for visibility
    fig.add_trace(go.Scatter(x=[origin[0], vec[0] * scale],y=[origin[1], vec[1] * scale],mode='lines',name=f'PC{i+1} {Lambda[i]:.2f}'))

fig.show()

Eigenvalues:
[6.48189768 0.96161457]

Mean squared projections:
[6.48189768 0.96161457]


<a class="anchor" id="interpretation"></a>
# 7. Interpretation

- PCA = **rotation + projection**
- Keeps:
    - Maximum variance
- Discards:
    - Low-variance directions (assumed less important)


<a class="anchor" id="notes"></a>
# 8. Additional Notes

- PCA:
    - Based on Linear algebra (Eigen decomposition, SVD)
    - Preserves variance
    - model with parameters
- Related methods:
    - LLE, ISOMAP, t-SNE, UMAP (nonlinear alternatives)
    - Preserve neighborhood relationships
    - model without parameters
- PCA is for **linear** data 
- For nonlinear **nonlinear data** use **kPCA**


<a class="anchor" id="library"></a>
# 9. PCA library

```python
# 1. Import PCA
from sklearn.decomposition import PCA

pca1 = PCA(
    n_components=2, # number of principal components
    whiten=False, # scale components to unit variance
    random_state=1414
)

X_pca = pca1.fit_transform(X)  # fit + transform
pca1.components_ # principal axes
pca1.explained_variance_ # variance per component
pca1.explained_variance_ratio_ # % variance explained
pca1.singular_values_ # singular values



# 2. Explained Variance Analysis
import numpy as np

pca2 = PCA().fit(X)

np.cumsum(pca2.explained_variance_ratio_) # cumulative variance



# 3. Choose Components by Variance Threshold
pca3 = PCA(n_components=0.95) # keep 95% variance
X_reduced = pca3.fit_transform(X)

pca3.n_components_ # actual number selected



# 4. Scree Plot
import matplotlib.pyplot as plt

pca4 = PCA().fit(X)

plt.plot(pca4.explained_variance_ratio_, marker="o")
plt.xlabel("Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot")
plt.show()



# 5. PCA with Standardization (important!)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca5 = PCA(n_components=2)
X_pca_scaled = pca5.fit_transform(X_scaled)



# 6. Inverse Transform (reconstruction)
X_approx = pca5.inverse_transform(X_pca_scaled)



# 7. Pipeline (recommended)
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2))
])

X_pipeline = pipeline.fit_transform(X)



# 8. PCA + Classification
from sklearn.ensemble import RandomForestClassifier

pipeline2 = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("clf", RandomForestClassifier())
])

pipeline2.fit(X, y)
pipeline2.score(X, y)


# 9. PCA for Visualization (2D)
pca9 = PCA(n_components=2)
X_2d = pca9.fit_transform(X)

plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection")
plt.show()


# 10. Compare different number of components
scores = []

for k in [1, 2, 3, 4, 5]:
    pca = PCA(n_components=k)
    X_k = pca.fit_transform(X)
    scores.append(sum(pca.explained_variance_ratio_))

print(scores)


# 11. PCA on Toy Data
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=100,
    n_features=5,
    n_informative=3,
    random_state=1415
)

pca11 = PCA(n_components=2)
X_pca = pca11.fit_transform(X)
```

In [ ]:
# Generate 3D data
np.random.seed(1443)
n = 100

mean = [3, 4, 5]
cov = [[6, 3, 2],
       [3, 2, 1],
       [2, 1, 2]]

X = np.random.multivariate_normal(mean, cov, size=n)

# Center data
X_centered = X - np.mean(X, axis=0)


# PCA via sklearn
pca = PCA(n_components=3)
Z = pca.fit_transform(X_centered)

eigenvalues = pca.explained_variance_
eigenvectors = pca.components_  # rows = eigenvectors


# Verify with projections
# projections onto eigenvectors
# (same as Z but recomputed explicitly)
projections = X_centered @ eigenvectors.T

mean_squared_projections = np.mean(projections**2, axis=0)

print("Eigenvalues (sklearn):")
print(eigenvalues)

print("\nMean squared projections:")
print(mean_squared_projections)


fig = px.scatter_3d(
    x=X_centered[:, 0],y=X_centered[:, 1],z=X_centered[:, 2],
    title="3D PCA: data with principal directions",
    labels={"x": "x1", "y": "x2", "z": "x3"})

# Add eigenvectors
origin = np.array([0, 0, 0])

for i in range(3):
    vec = eigenvectors[i]
    scale = np.sqrt(eigenvalues[i]) * 3  # scale for visibility
    fig.add_trace(go.Scatter3d(
        x=[origin[0], vec[0] * scale],y=[origin[1], vec[1] * scale],z=[origin[2], vec[2] * scale],
        mode='lines',line=dict(width=6),name=f'PC{i+1} (λ={eigenvalues[i]:.2f})'))

fig.show()


# Plot in 2D (first 2 PCs)
fig2 = px.scatter(x=Z[:, 0],y=Z[:, 1],title="Projection onto first two principal components",labels={"x": "PC1", "y": "PC2"})

fig2.show()

# Reconstruct using only first 2 PCs
Z_2 = Z[:, :2]
V_2 = eigenvectors[:2]

X_approx = Z_2 @ V_2

# compare with original centered data
print("Reconstruction error:",np.mean(np.linalg.norm(X_centered - X_approx, axis=1)**2))

Eigenvalues (sklearn):
[8.46181929 1.53921194 0.45563657]

Mean squared projections:
[8.37720109 1.52381982 0.4510802 ]


Reconstruction error: 0.4510802043839277
